# v0.9.0 -- Transactions: QuerySet routing & strategies

v0.9.0 extends transactions in two ways:

1. **`objects(tx=)`** routes `QuerySet` reads (`get`, `first`, `all`, `exec`, `count`, `exists`,
   aggregations) and bulk operations (`bulk_create`, `bulk_update`, `bulk_delete`) through the
   transaction.
2. **Two strategies**, auto-selected by `transaction()`:
   - `InteractiveTransaction` -- WebSocket + SurrealDB 3.x, using the SDK's native
     `begin()/commit()/cancel()` API. Reads see uncommitted writes; `save(tx=)` supports
     auto-generated ids.
   - `BufferedTransaction` -- HTTP, or WebSocket on SurrealDB 2.6.x: the v0.8.0 batching model.

We connect over WebSocket, then probe the server so every cell runs on 2.6.x too.

## 1. Connect (WebSocket) and probe the strategy

In [1]:
import os
from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
# WebSocket (.../rpc) lets transaction() use SurrealDB 3.x native interactive transactions.
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root", password="root",
    namespace="examples", database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


In [2]:
import contextlib


async def native_tx_supported() -> bool:
    """True if the server exposes the SDK's native transaction RPC (SurrealDB 3.x)."""
    client = await SurrealDBConnectionManager.get_client()
    try:
        txn = await client.begin()
    except Exception:
        return False
    with contextlib.suppress(Exception):
        await client.cancel(txn)
    return True


interactive = await native_tx_supported()
print("Native interactive transactions available (SurrealDB 3.x):", interactive)

Native interactive transactions available (SurrealDB 3.x): True


## 2. Define a model and reset the table

In [3]:
import contextlib

from surreal_orm_lite import BaseSurrealModel, SurrealConfigDict


class Member(BaseSurrealModel):
    model_config = SurrealConfigDict(primary_key="id")
    id: str | None = None
    name: str
    role: str = "guest"


client = await SurrealDBConnectionManager.get_client()
with contextlib.suppress(Exception):
    await client.query("DELETE Member;", {})
print("table reset")

table reset


## 3. Which strategy did `transaction()` pick?

The concrete class is chosen from the connection (URL scheme + server version).

In [4]:
async with SurrealDBConnectionManager.transaction() as tx:
    print("strategy:", type(tx).__name__)
    print("is_interactive:", tx.is_interactive)

strategy: InteractiveTransaction
is_interactive: True


## 4. Reads inside the transaction see uncommitted writes (interactive)

On the interactive strategy a write made earlier in the block is visible to a later read in the **same** transaction. On a buffered transaction reads aren't supported, so we explain that instead.

In [5]:
if interactive:
    async with SurrealDBConnectionManager.transaction() as tx:
        await Member(id="m1", name="Ada", role="member").save(tx=tx)
        found = await Member.objects(tx=tx).filter(role="member").exec()
        print("seen inside the still-open tx:", [m.name for m in found])
else:
    print("Buffered strategy: reads inside a transaction are unsupported and raise; "
          "the writes still commit atomically at block exit.")

seen inside the still-open tx: ['Ada']


## 5. Bulk update routed through the transaction

Promote every guest to member atomically. The interactive strategy returns the real affected count; the buffered strategy can't know it before commit and returns `0` (same contract as `bulk_update` outside a tx).

In [6]:
with contextlib.suppress(Exception):
    await client.query("DELETE Member;", {})
await Member(id="g1", name="Grace", role="guest").save()
await Member(id="g2", name="Lin", role="guest").save()

async with SurrealDBConnectionManager.transaction() as tx:
    n = await Member.objects(tx=tx).filter(role="guest").bulk_update(role="member")
    print("affected count reported:", n)

rows = await client.query("SELECT name, role FROM Member ORDER BY name;", {})
print("after commit:", {r["name"]: r["role"] for r in rows})

affected count reported: 2
after commit: {'Grace': 'member', 'Lin': 'member'}


## 6. Auto-generated ids inside a transaction (interactive only)

v0.8.0 required an explicit id inside a transaction. The interactive strategy lifts that: `save(tx=)` with no id gets a server-generated one.

In [7]:
if interactive:
    m = Member(name="Anonymous")  # no id
    async with SurrealDBConnectionManager.transaction() as tx:
        await m.save(tx=tx)
    print("assigned id:", m.id)
else:
    print("Buffered strategy: an explicit id is still required inside a transaction.")

assigned id: Member:1907omkygxp9w6gg9sgb


## 7. Rollback still protects bulk operations

In [8]:
from surreal_orm_lite import SurrealDbError

before = len(await client.query("SELECT * FROM Member;", {}))
try:
    async with SurrealDBConnectionManager.transaction() as tx:
        await Member.objects(tx=tx).bulk_delete()  # would wipe the table
        raise RuntimeError("changed my mind")
except RuntimeError as exc:
    print("rolled back:", exc)
after = len(await client.query("SELECT * FROM Member;", {}))
print(f"rows before={before} after={after} (unchanged)")

rolled back: changed my mind
rows before=3 after=3 (unchanged)


## 8. Cleanup

In [9]:
await client.query("DELETE Member;", {})
await SurrealDBConnectionManager.close_connection()
print("done")

done
